Creating gold dimension and fact tables. 

In [68]:
from pyspark.sql.functions import ( 
    col, year, month, quarter, dayofweek, date_format, coalesce, expr, to_date, 
    sum as spark_sum, count, avg, round as spark_round, lit 
) 
from pyspark.sql.types import IntegerType 
from pyspark.sql import functions as F 

StatementMeta(, 44fb0acf-995d-4598-89d6-cbafee8f261e, 70, Finished, Available, Finished, False)

In [69]:
#Loading tables as DataFrames
d = spark.table("silver_sales_order_detail")
h = spark.table("silver_sales_order_header")
p = spark.table("silver_dim_product")

#Building fact table Sales
df_fact = (
    d.join(h, d.SalesOrderID == h.SalesOrderID, "inner")
     .join(
         p,
         (d.ProductID == p.ProductID) &
         (h.OrderDate.between(p.effective_date, p.expiry_date)),
         "inner"
     )
     .filter(h.OrderDate.isNotNull())
     .select(
         d.SalesOrderDetailID,
         h.SalesOrderID,
         h.OrderDate,
         d.ProductID,
         h.CustomerID,
         h.TerritoryID,
         d.OrderQty,
         d.UnitPrice,
         d.LineTotal.alias("revenue"),
         h.TotalDue,
         p.ListPrice.alias("list_price_at_sale"),
         p.StandardCost.alias("cost_at_sale"),
         ((d.UnitPrice - p.StandardCost) * d.OrderQty).alias("gross_profit")
     )
)

#Writing to Delta table
df_fact.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("gold_fact_sales")

print(f"Fact rows: {df_fact.count()}")

StatementMeta(, 44fb0acf-995d-4598-89d6-cbafee8f261e, 71, Finished, Available, Finished, False)

Fact rows: 121317


In [1]:
#Gold dim_product

#Reading the silver table
silver_p_df = spark.table("silver_dim_product")

#Selecting the required columns
gold_p_df = silver_p_df.select(
    "ProductID", "Name", "ProductNumber", "ProductSubcategoryID","ListPrice", "StandardCost", "Color", "Size",
    "effective_date", "expiry_date", "is_current", "record_version"
)

gold_p_current_df = gold_p_df.filter("is_current = true")

#Overwrite or create the gold table
gold_p_current_df.write.mode("overwrite").saveAsTable("gold_dim_product")

gold_p_df.count()

StatementMeta(, 8d67852d-6609-4846-a2b1-114fe6d92ee2, 3, Finished, Available, Finished, False)

519

In [71]:
#Gold dim_territory
silver_t_df = spark.table("silver_dim_territory")

gold_t_df = silver_t_df.select(
    "TerritoryID", silver_t_df["Name"].alias("territory_name"),
    "CountryRegionCode", silver_t_df["Group"].alias("sales_group")
) 
gold_t_df.write.mode("overwrite").saveAsTable("gold_dim_territory")

gold_t_df.count()

StatementMeta(, 44fb0acf-995d-4598-89d6-cbafee8f261e, 73, Finished, Available, Finished, False)

10

In [72]:
#Gold dim_customer 
silver_c_df = spark.table("silver_dim_customer")

gold_c_df = silver_c_df.select(
    "CustomerID", "PersonID", "StoreID", "TerritoryID"
)

gold_c_df.write.mode("overwrite").saveAsTable("gold_dim_customer")

gold_c_df.count()

StatementMeta(, 44fb0acf-995d-4598-89d6-cbafee8f261e, 74, Finished, Available, Finished, False)

19820

In [75]:
#Gold dim_date 

#Defining start and end dates
start_date = "2011-01-01"
end_date = "2026-04-30"

#ComputING the number of days between start and end
days_df = spark.range(0, (spark.sql(f"SELECT datediff('{end_date}', '{start_date}') + 1").collect()[0][0]))

#GeneratING date column by adding days to start date
date_range = days_df.select(
    (to_date(expr(f"date_add('{start_date}', cast(id as int))"))).alias("date")
)
date_range.show(5)

df_date = ( 
    date_range 
    .withColumn('DateKey',date_format(col('date'), 'yyyyMMdd').cast(IntegerType())) 
    .withColumn('Year',year(col('date'))) 
    .withColumn('Quarter',quarter(col('date'))) 
    .withColumn('Month',month(col('date'))) 
    .withColumn('MonthName',date_format(col('date'), 'MMMM')) 
    .withColumn('WeekDay',dayofweek(col('date'))) 
    .withColumn('DayName',date_format(col('date'), 'EEEE')) 
    .withColumn('FiscalYear',(year(col('date')) + (month(col('date')) >= 7).cast(IntegerType()))) 
) 
df_date.write.format('delta').mode('overwrite').saveAsTable('gold_dim_date') 

print('Gold layer complete.') 

StatementMeta(, 44fb0acf-995d-4598-89d6-cbafee8f261e, 77, Finished, Available, Finished, False)

+----------+
|      date|
+----------+
|2011-01-01|
|2011-01-02|
|2011-01-03|
|2011-01-04|
|2011-01-05|
+----------+
only showing top 5 rows

Gold layer complete.
